# ingestion_and_preprocessing

## PDF PARSING

In [4]:
# Library Imports
from IPython.display import JSON, display
import json
from unstructured.partition.pdf import partition_pdf

c:\Users\SHREY\anaconda3\envs\uc_rag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### HI_RES VS OCR_ONLY PARSING STRATEGY

##### HI-RES

In [ ]:
elements = partition_pdf(filename=r"dataset\Cryptography_CSE3703.pdf",
                         strategy="hi_res",
                         infer_table_structure=True,
                         extract_images_in_pdf=False,
                         languages=['en'])
print(len(elements))

42


In [38]:
element_dict = [el.to_dict() for el in elements]
with open("hi_res_output.json", "w", encoding="utf-8") as f:
    json.dump(element_dict, f, indent=2, ensure_ascii=False)

##### OCR-ONLY

In [39]:
elements = partition_pdf(filename="dataset\Cryptography_CSE3703.pdf",
                         strategy="ocr_only",
                         infer_table_structure=True,
                         extract_images_in_pdf=False,
                         languages=['en'])
print(len(elements))

72


In [40]:
element_dict = [el.to_dict() for el in elements]
with open("ocr_only_output.json", "w", encoding="utf-8") as f:
    json.dump(element_dict, f, indent=2, ensure_ascii=False)

##### conclusion

<p>hi_res is much better at recognizing tables even though the correctly recognized text sometimes gets jumbled in the text as html column, overall it seems to be a fair tradeoff due to the much better tagging of tables in it and recognition of structure</p>

#### QUALITY CHECKS

In [41]:
import json

# Load your JSON data
with open('hi_res_output.json', 'r') as f:
    data = json.load(f)

# 1. Create a lookup map for quick access: {element_id: text}
# This allows us to find the parent text in O(1) time
id_to_text = {item['element_id']: item.get('text', '') for item in data}

# 2. Extract all unique parent_ids from the metadata
# We use a set to avoid duplicates and handle potential missing keys
parent_ids = {
    item['metadata']['parent_id'] 
    for item in data 
    if 'metadata' in item and 'parent_id' in item['metadata']
}

print(f"--- Found {len(parent_ids)} unique Parent IDs ---")

# 3. Print the text for each parent_id found in our lookup map
for pid in parent_ids:
    if pid in id_to_text:
        print(f"\nParent ID: {pid}")
        print(f"Parent Text: {id_to_text[pid]}")
        print("-" * 30)
    else:
        # This happens if the parent_id refers to an element not in this JSON list
        print(f"\nParent ID: {pid} (Text not found in this file)")

--- Found 7 unique Parent IDs ---

Parent ID: a121618a2ac1f98bf630fa223f16ea42
Parent Text: Topics of the course:
------------------------------

Parent ID: e2b54f40ac29cde593e118eefa8f87c8
Parent Text: CO/PO Mapping:
------------------------------

Parent ID: 5e14ae08f2eca7d992901f296175dfd5
Parent Text: Assessment Pattern:
------------------------------

Parent ID: 055cf135f5684441b239b528afc9ec1a
Parent Text: Student Responsibilities:
------------------------------

Parent ID: 35975b2fac6350045b1b1f3016a1ad04
Parent Text: Reference Books:
------------------------------

Parent ID: 5e3e3ca0ededa39a6b2f7e928c180031
Parent Text: Learning Resources:
------------------------------

Parent ID: d557b3ba5f1537479fc9e399238638f7
Parent Text: Experiential Learning Component (20 %):
------------------------------


In [46]:
import json

with open('hi_res_output.json', 'r') as f:
    data = json.load(f)

def has_extra_whitespaces(text):
    if not text:
        return False
    # Compare original text to text stripped of extra whitespace
    cleaned = " ".join(text.split())
    return text != cleaned

# Track statistics
total_elements = len(data)
num_elements_with_extra_whitespace = 0
examples = []

for item in data:
    original_text = item.get('text', '')
    if has_extra_whitespaces(original_text):
        num_elements_with_extra_whitespace += 1
        # Save the first 3 examples to see what's wrong
        if len(examples) < 3:
            examples.append(original_text)

# Print Report
print(f"Total Elements: {total_elements}")
print(f"Elements with Extra Whitespace: {num_elements_with_extra_whitespace}")
print(f"Clean Elements: {total_elements - num_elements_with_extra_whitespace}")
print(f"Percentage of Text that Needs to Be Cleaned: {(num_elements_with_extra_whitespace / total_elements) * 100:.2f}%")

if examples:
    print("\n--- Examples of Data that required cleaning ---")
    for i, ex in enumerate(examples, 1):
        # repr() shows hidden characters like \n or \t
        print(f"{i}. {repr(ex)}")

Total Elements: 42
Elements with Extra Whitespace: 0
Clean Elements: 42
Percentage of Text that Needs to Be Cleaned: 0.00%


In [47]:
import json

with open('hi_res_output.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

def is_missing_text(text):
    if text is None:
        return True
    if isinstance(text, str):
        return text.strip() == ''
    return True

total_elements = len(data)
num_elements_missing_text = 0
examples_with_missing_text = []

for item in data:
    text = item.get('text', None)
    if is_missing_text(text):
        num_elements_missing_text += 1
        if len(examples_with_missing_text) < 5:
            examples_with_missing_text.append({
                "element_id": item.get("element_id"),
                "page": item.get("metadata", {}).get("page_number"),
                "text_repr": repr(text)
            })

print("--- Missing / Empty Text Report ---")
print(f"Total elements: {total_elements}")
print(f"Elements with missing/empty text: {num_elements_missing_text}")
print(f"Clean elements: {total_elements - num_elements_missing_text}")
print(f"Percentage missing: {(num_elements_missing_text / total_elements) * 100:.2f}%")

if examples_with_missing_text:
    print("\nExamples:")
    for ex in examples_with_missing_text:
        print(f"- id: {ex['element_id']}, page: {ex['page']}, text: {ex['text_repr']}")

--- Missing / Empty Text Report ---
Total elements: 42
Elements with missing/empty text: 0
Clean elements: 42
Percentage missing: 0.00%


#### CLEANING

Handles both whitespace normalization and the removal of empty or null text elements.



In [6]:
import json
import os

def clean_json_elements(file_path):
    """
    Cleans a JSON file by:
    1. Removing elements with null, empty, or whitespace-only 'text'.
    2. Stripping extra whitespaces from valid 'text' fields.
    Updates the file in-place.
    """
    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found.")
        return

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    original_count = len(data)
    
    # Define cleaning and filtering logic
    cleaned_data = []
    for item in data:
        text = item.get('text')
        
        # 1. Skip if text is None, not a string, or just whitespace
        if text is None or not isinstance(text, str) or text.strip() == '':
            continue
            
        # 2. Remove extra internal whitespaces (e.g., "  " -> " ")
        # and leading/trailing whitespace
        item['text'] = " ".join(text.split())
        
        cleaned_data.append(item)

    # 3. Save the cleaned data back to the same path
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(cleaned_data, f, indent=2, ensure_ascii=False)

    removed = original_count - len(cleaned_data)
    print(f"File: {file_path}")
    print(f"Elements processed: {original_count}")
    print(f"Null/Void elements removed: {removed}")
    print(f"Remaining elements: {len(cleaned_data)}")

In [7]:
clean_json_elements(r"C:\Users\SHREY\Desktop\uc_rag\hi_res_output.json")

File: C:\Users\SHREY\Desktop\uc_rag\hi_res_output.json
Elements processed: 42
Null/Void elements removed: 0
Remaining elements: 42




### Key Logic used:
- **`" ".join(text.split())`**: This is a robust way to remove all types of extra whitespace (tabs, newlines, multiple spaces) and collapse them into a single space.
- **Filtering**: It uses a list comprehension (effectively) to only keep items where `text.strip()` is not an empty string.
- **In-place**: The file is overwritten with the `cleaned_data` list.

## DOC PARSING

In [28]:
# Library Imports
from IPython.display import JSON, display
import json
from unstructured.partition.doc import partition_doc

In [ ]:
elements = partition_doc(filename=r"dataset\Network Security_CSE3714.doc", languages=['en'])
print(len(elements))

47


In [30]:
element_dict = [el.to_dict() for el in elements]
with open("doc_output.json", "w", encoding="utf-8") as f:
    json.dump(element_dict, f, indent=2, ensure_ascii=False)

##### QUALITY CHECKS

In [9]:
import json

with open('doc_output.json', 'r') as f:
    data = json.load(f)

def has_extra_whitespaces(text):
    if not text:
        return False
    # Compare original text to text stripped of extra whitespace
    cleaned = " ".join(text.split())
    return text != cleaned

# Track statistics
total_elements = len(data)
num_elements_with_extra_whitespace = 0
examples = []

for item in data:
    original_text = item.get('text', '')
    if has_extra_whitespaces(original_text):
        num_elements_with_extra_whitespace += 1
        # Save the first 3 examples to see what's wrong
        if len(examples) < 3:
            examples.append(original_text)

# Print Report
print(f"Total Elements: {total_elements}")
print(f"Elements with Extra Whitespace: {num_elements_with_extra_whitespace}")
print(f"Clean Elements: {total_elements - num_elements_with_extra_whitespace}")
print(f"Percentage of Text that Needs to Be Cleaned: {(num_elements_with_extra_whitespace / total_elements) * 100:.2f}%")

if examples:
    print("\n--- Examples of Data that required cleaning ---")
    for i, ex in enumerate(examples, 1):
        # repr() shows hidden characters like \n or \t
        print(f"{i}. {repr(ex)}")

Total Elements: 47
Elements with Extra Whitespace: 0
Clean Elements: 47
Percentage of Text that Needs to Be Cleaned: 0.00%


In [38]:
import json

with open('hi_res_output.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

def is_missing_text(text):
    if text is None:
        return True
    if isinstance(text, str):
        return text.strip() == ''
    return True

total_elements = len(data)
num_elements_missing_text = 0
examples_with_missing_text = []

for item in data:
    text = item.get('text', None)
    if is_missing_text(text):
        num_elements_missing_text += 1
        if len(examples_with_missing_text) < 5:
            examples_with_missing_text.append({
                "element_id": item.get("element_id"),
                "page": item.get("metadata", {}).get("page_number"),
                "text_repr": repr(text)
            })

print("--- Missing / Empty Text Report ---")
print(f"Total elements: {total_elements}")
print(f"Elements with missing/empty text: {num_elements_missing_text}")
print(f"Clean elements: {total_elements - num_elements_missing_text}")
print(f"Percentage missing: {(num_elements_missing_text / total_elements) * 100:.2f}%")

if examples_with_missing_text:
    print("\nExamples:")
    for ex in examples_with_missing_text:
        print(f"- id: {ex['element_id']}, page: {ex['page']}, text: {ex['text_repr']}")

--- Missing / Empty Text Report ---
Total elements: 42
Elements with missing/empty text: 0
Clean elements: 42
Percentage missing: 0.00%


##### CLEANING

In [8]:
clean_json_elements(r"C:\Users\SHREY\Desktop\uc_rag\doc_output.json")

File: C:\Users\SHREY\Desktop\uc_rag\doc_output.json
Elements processed: 47
Null/Void elements removed: 0
Remaining elements: 47


## COMPARING DOC AND PDF JSON TAGS FOR NORMALIZATION

In [31]:
import json
import os
from collections import Counter

def analyze_json_file(file_path):
    # Check if file exists
    if not os.path.exists(file_path):
        print(f"Error: The file '{file_path}' was not found.")
        return

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        if not isinstance(data, list):
            print("Error: JSON top-level structure must be a list of objects.")
            return

        total_elements = len(data)
        
        # 1. Map all keys found in each element
        all_key_occurrences = [set(element.keys()) for element in data]
        
        # 2. Count how many times each key appears across all elements
        key_counts = Counter()
        for key_set in all_key_occurrences:
            key_counts.update(key_set)
            
        # 3. Categorize keys
        all_unique_tags = sorted(list(key_counts.keys()))
        common_tags = [tag for tag, count in key_counts.items() if count == total_elements]
        uncommon_tags = [tag for tag, count in key_counts.items() if count < total_elements]

        # --- Display Results ---
        print(f"Successfully loaded {total_elements} elements from: {file_path}\n")
        
        print("### 1. ALL UNIQUE TAGS")
        print(f"Total count: {len(all_unique_tags)}")
        print(all_unique_tags)
        
        print("\n### 2. COMMON TAGS (Present in ALL elements)")
        print(f"Total count: {len(common_tags)}")
        print(common_tags)
        
        print("\n### 3. UNCOMMON TAGS (Missing from at least one element)")
        print(f"Total count: {len(uncommon_tags)}")
        print(uncommon_tags)

    except json.JSONDecodeError:
        print("Error: Failed to decode JSON. Check if the file format is valid.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [32]:
path_to_file = 'doc_output.json' 
analyze_json_file(path_to_file)

Successfully loaded 47 elements from: doc_output.json

### 1. ALL UNIQUE TAGS
Total count: 4
['element_id', 'metadata', 'text', 'type']

### 2. COMMON TAGS (Present in ALL elements)
Total count: 4
['element_id', 'text', 'metadata', 'type']

### 3. UNCOMMON TAGS (Missing from at least one element)
Total count: 0
[]


In [33]:
path_to_file = 'hi_res_output.json' 
analyze_json_file(path_to_file)

Successfully loaded 42 elements from: hi_res_output.json

### 1. ALL UNIQUE TAGS
Total count: 4
['element_id', 'metadata', 'text', 'type']

### 2. COMMON TAGS (Present in ALL elements)
Total count: 4
['element_id', 'text', 'metadata', 'type']

### 3. UNCOMMON TAGS (Missing from at least one element)
Total count: 0
[]


In [34]:
import json
import os
from collections import Counter

def analyze_metadata_tags(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found.")
        return

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # Filter elements that actually contain a 'metadata' key and it's a dict
        metadata_elements = [el['metadata'] for el in data if isinstance(el.get('metadata'), dict)]
        total_with_metadata = len(metadata_elements)
        
        if total_with_metadata == 0:
            print("No 'metadata' dictionaries found in the elements.")
            return

        # 1. Map internal keys found within 'metadata'
        meta_key_occurrences = [set(m.keys()) for m in metadata_elements]
        
        # 2. Count frequency of these internal tags
        meta_counts = Counter()
        for s in meta_key_occurrences:
            meta_counts.update(s)
            
        # 3. Categorize internal tags
        all_meta_tags = sorted(list(meta_counts.keys()))
        common_meta = [tag for tag, count in meta_counts.items() if count == total_with_metadata]
        uncommon_meta = [tag for tag, count in meta_counts.items() if count < total_with_metadata]

        # --- Display Results ---
        print(f"Analysis of 'metadata' tags (Found in {total_with_metadata}/{len(data)} elements):")
        print("=" * 60)
        
        print(f"\n[+] ALL UNIQUE METADATA TAGS ({len(all_meta_tags)}):")
        print(all_meta_tags)
        
        print(f"\n[+] COMMON METADATA TAGS (Present in all metadata objects):")
        print(common_meta if common_meta else "None")
        
        print(f"\n[+] UNCOMMON METADATA TAGS (Missing from some metadata objects):")
        if uncommon_meta:
            # We add a bit of detail: which tags appear how often
            for tag in sorted(uncommon_meta):
                print(f" - {tag}: appears in {meta_counts[tag]}/{total_with_metadata} metadata objects")
        else:
            print("None")

    except Exception as e:
        print(f"An error occurred: {e}")

In [35]:
path_to_file = 'doc_output.json'
analyze_metadata_tags(path_to_file)

Analysis of 'metadata' tags (Found in 47/47 elements):

[+] ALL UNIQUE METADATA TAGS (9):
['category_depth', 'emphasized_text_contents', 'emphasized_text_tags', 'file_directory', 'filename', 'filetype', 'languages', 'last_modified', 'text_as_html']

[+] COMMON METADATA TAGS (Present in all metadata objects):
['last_modified', 'filetype', 'filename', 'languages', 'file_directory']

[+] UNCOMMON METADATA TAGS (Missing from some metadata objects):
 - category_depth: appears in 41/47 metadata objects
 - emphasized_text_contents: appears in 23/47 metadata objects
 - emphasized_text_tags: appears in 23/47 metadata objects
 - text_as_html: appears in 6/47 metadata objects


In [36]:
path_to_file = 'hi_res_output.json' 
analyze_metadata_tags(path_to_file)

Analysis of 'metadata' tags (Found in 42/42 elements):

[+] ALL UNIQUE METADATA TAGS (11):
['coordinates', 'detection_class_prob', 'file_directory', 'filename', 'filetype', 'is_extracted', 'languages', 'last_modified', 'page_number', 'parent_id', 'text_as_html']

[+] COMMON METADATA TAGS (Present in all metadata objects):
['is_extracted', 'coordinates', 'last_modified', 'page_number', 'filetype', 'filename', 'languages', 'file_directory']

[+] UNCOMMON METADATA TAGS (Missing from some metadata objects):
 - detection_class_prob: appears in 38/42 metadata objects
 - parent_id: appears in 27/42 metadata objects
 - text_as_html: appears in 7/42 metadata objects


## NORMALIZATION AND SCHEMA

In [2]:
def normalize_element(el, source):
    meta = el.get("metadata", {})
    filename = meta.get("filename", "")

    coursename = None
    coursecode = None

    if filename and "_" in filename:
        parts = filename.split("_", 1)
        coursename = parts[0].strip()
        coursecode = parts[1].strip()
    else:
        coursename = filename  # fallback
        coursecode = None

    return {
        "element_id": el.get("element_id"),
        "type": el.get("type"),
        "text": el.get("text"),

        # --- unified metadata ---
        "source": source,
        "filename": filename,
        "coursename": coursename,
        "coursecode": coursecode,

        # page logic
        "page": meta.get("page_number") if source == "pdf" else None,

        # --- structure ---
        "parent_id": meta.get("parent_id"),

        # --- tables ---
        "table_html": meta.get("text_as_html") if el.get("type") == "Table" else None,
    }

In [ ]:
import json
# 1. Load the JSON files
with open('hi_res_output.json', 'r', encoding='utf-8') as f:
    hi_res_data = json.load(f)

with open('doc_output.json', 'r', encoding='utf-8') as f:
    doc_data = json.load(f)

# 2. Normalize the data
normalized_pdf = [normalize_element(el, "pdf") for el in hi_res_data]
normalized_doc = [normalize_element(el, "doc") for el in doc_data]

# 3. Save to new JSON files
with open('normalized_pdf.json', 'w', encoding='utf-8') as f:
    json.dump(normalized_pdf, f, indent=2, ensure_ascii=False)

with open('normalized_doc.json', 'w', encoding='utf-8') as f:
    json.dump(normalized_doc, f, indent=2, ensure_ascii=False)

print(f"Normalized {len(normalized_pdf)} PDF elements and {len(normalized_doc)} DOC elements.")

Normalized 42 PDF elements and 47 DOC elements.


## FINAL REMARKS

DATA INGESTION -> PARSING -> SERIALIZATION -> CLEANING -> NORMALIZATION
done
NEXT STEP: MAKE THIS MODULAR PIPELINE